## Pre-Process Stakeholder Data

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from analysis.utils import get_embeddings


def build_stakeholder_df(path="./data/partial.json"):
    with open(path) as f:
        data = json.loads(f.read())

    records = []
    for item in data:
        study_id = item["StudyID"]
        
        for stakeholder in item["Stakeholders"]:
            records.append({
                "StudyID": study_id,
                "StudyType": item["Type"].split("-")[0],
                "Stage": "Brainstorming",
                "Type": "Stakeholder",
                "Stakeholder": stakeholder,
                "Content": ""
            })
        
        for stage in ["Before", "After"]:
            for entry_type in ["Harms", "Mitigations"]:
                entries = item.get(stage, {}).get(entry_type, [])
                for entry in entries:
                    stakeholder = entry.get("Stakeholder", "Unknown")
                    stakeholders = stakeholder.split(", ")
                    for stakeholder in stakeholders:
                        records.append({
                            "StudyID": study_id,
                            "StudyType": item["Type"].split("-")[0],
                            "Stage": stage,
                            "Type": entry_type[:-1],  # Remove plural "s" from "Harms"/"Mitigations"
                            "Stakeholder": stakeholder,
                            "Content": entry.get("Harm") or entry.get("Mitigation")
                        })

    df = pd.DataFrame(records)

    df['StudyID'] = df['StudyID'].astype(str) + "-" + df['StudyType']

    df['embedding'] = get_embeddings(df['Stakeholder'])

    return df

df = build_stakeholder_df()
df

In [ ]:
df.to_csv("data/stakeholders_long.csv", index=False)

## Pre-Process Story Data

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from analysis.utils import get_embeddings
from typing import List
import dspy
from analysis.utils import batch_inference, LM_DICT, use_lm

class NormalizeHarms(dspy.Signature):
    """You are given a user-written text about their envisioned harms of an AI system. Clean up the text and normalize it to a standard format. 
There can be multiple harms in the text, and you should separate them into a list.

Examples:
- Misinterpretation of Minority Culture Elements
- Demeaning Language in Report Narratives"""
    
    harm: str = dspy.InputField(desc="Description of a harm")
    harm_list: List[str] = dspy.OutputField(desc="Normalized harm description")


def build_story_df(harm_path="./data/RAI_taxonomy.json", story_path="./data/RAI_story_taxonomy.json"):
    with open(harm_path) as f:
        data = json.loads(f.read())
    df = pd.DataFrame(data)

    df_melted = df.melt(id_vars=["StudyID", "Type"], value_vars=["Before", "After"],
                    var_name="Condition", value_name="Harm")
    
    df_melted["Harm"] = df_melted["Harm"].apply(lambda x: x.get("Harm") if isinstance(x, dict) else None)

    with open(story_path) as f:
        data = json.loads(f.read())
    df = pd.DataFrame(data)

    df_exploded = df.explode('Stories', ignore_index=True)

    # Extract HarmStory from each dict
    df_exploded['HarmStory'] = df_exploded['Stories'].apply(lambda x: x.get('HarmStory') if isinstance(x, dict) else None)

    # Drop original Stories column if no longer needed
    df_exploded = df_exploded.drop(columns=['Stories'])

    df_exploded['Condition'] = 'Story'
    df_exploded = df_exploded.rename(columns={'StudyID': 'StudyID', 'Type': 'Type', 'HarmStory': 'Harm'})

    final_df = pd.concat([df_melted, df_exploded], ignore_index=True)
    final_df['Type'] = final_df['Type'].apply(lambda x: x.split('-')[0])
    final_df['StudyID'] = final_df['StudyID'].astype(str) + "-" + final_df['Type']

    final_df = final_df[~(final_df['Harm'] == "")]
    return final_df

def normalize_harms_df(final_df):
    normalize_harms = use_lm(LM_DICT['gpt-4o'])(dspy.Predict(NormalizeHarms))

    results = batch_inference(
        normalize_harms,
        [{"harm": harm} for harm in final_df['Harm'].tolist()],
    )

    final_df['Normalized_Harm'] = [r['harm_list'] for r in results]
    final_df = final_df.explode('Normalized_Harm', ignore_index=True)
    final_df['embedding'] = get_embeddings(final_df['Normalized_Harm'])
    return final_df

final_df = build_story_df()
final_df = normalize_harms_df(final_df)
final_df

In [ ]:
final_df.to_csv("./data/harm_long.csv", index=False)

## Visualization

In [ ]:
import pandas as pd
import numpy as np
import ast
from sklearn.manifold import TSNE

import textwrap
import umap

import dash
from dash import dcc, html
import plotly.express as px

# Load or copy your DataFrame
def generate_tsne_data(df):
    df['embedding'] = df['embedding'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

    # Parse the embedding column if needed
    df['embedding'] = df['embedding'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

    # Convert embeddings to NumPy array
    X = np.array(df['embedding'].tolist())

    # Run t-SNE
    tsne = umap.UMAP(random_state=42)
    tsne_result = tsne.fit_transform(X)

    # Add results to DataFrame
    df['tsne-2d-one'] = tsne_result[:, 0]
    df['tsne-2d-two'] = tsne_result[:, 1]
    return df

df = pd.read_csv("./data/harm_long.csv")
def wrap_text(text, width=40):
    if isinstance(text, str):
        return '<br>'.join(textwrap.wrap(text, width=width))
    return text
df['Normalized_Harm'] = df['Normalized_Harm'].apply(wrap_text)

df = generate_tsne_data(df)
fig = px.scatter(
    df,
    x='tsne-2d-one',
    y='tsne-2d-two',
    color='Condition',
    hover_data=['Type', 'Normalized_Harm'],
    facet_col='StudyID',  # <-- This creates a subplot per StudyID
    facet_col_wrap=6,
    title='Harm Visualization',
    width=1200,
    height=500
)

df = pd.read_csv("./data/stakeholders_long.csv")
df = generate_tsne_data(df)
fig2 = px.scatter(
    df,
    x='tsne-2d-one',
    y='tsne-2d-two',
    color='Stage',  # Vary shapes by StudyType
    hover_data=['Type', 'Stakeholder'],
    facet_col='StudyID',  # <-- This creates a subplot per StudyID
    facet_col_wrap=6,
    title='Stakeholder Visualization',
    width=1200,
    height=500
)


# Build Dash app
app = dash.Dash(__name__)
app.layout = html.Div(children=[
    dcc.Graph(figure=fig),
    dcc.Graph(figure=fig2)
])

if __name__ == '__main__':
    app.run(debug=True)